Merging multiple tif files

In [ ]:
!uv add matplotlib

In [2]:
# Uncomment only if rasterio is not already installed.
# %pip install rasterio matplotlib numpy

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.plot import plotting_extent

In [ ]:
Year = 2014

In [ ]:
from pathlib import Path

parent_folder = Path(
    "/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/county_year_wise_VI_tiff/2014-2019/DE/"
)

# Automatically collect only the Wisconsin 2014 export tiles
tif_files = sorted(
    parent_folder.glob(
        # f"soybeans_DE_{Year}_vi_stack_minimal-[0-9]*-[0-9]*.tif"
        f"soybeans_DE_{Year}_*.tif"
    )
)

if not tif_files:
    raise FileNotFoundError(
        f"No matching TIFF files were found in:\n{parent_folder}"
    )

print(f"Number of TIFF tiles found: {len(tif_files)}")

for index, path in enumerate(tif_files, start=1):
    print(f"{path.name}")

In [ ]:
# tif_files= [
#     Path("/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/soybeans_DE_2014_10005_Sussex_pseudo_yield.tif"),
#     Path("/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/soybeans_DE_2014_10001_Kent_pseudo_yield.tif"),
#     Path("/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/soybeans_DE_2014_10003_New_pseudo_yield.tif"),
# ]

### Merging the pseudo yield files for the counties in the state to get a one State-Year Pseudo Yield file:


In [4]:
# YEAR = 2019
# STATE = ""

for STATE in ['MN']:
    for YEAR in [2023]:
        parent_folder = Path("/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/Intermediate")
        tif_files = sorted(
            parent_folder.glob(
                f"soybeans_{STATE}_{YEAR}_*.tif"
            )
        )

        print("Number of files found:", len(tif_files))
        print("Files found:")
        for file in tif_files:
            print(file)

        from pathlib import Path

        # ============================================================
        # USER SETTINGS
        # ============================================================

        # parent_folder = Path(
        #     "/Users/samarranjit/Library/CloudStorage/"
        #     "GoogleDrive-samarranjit1234@gmail.com/My Drive/"
        #     "GEE_SOYBEAN_VI_EXPORTS"
        # )

        # # Complete list of the 17 Illinois 2014 TIFF tiles
        # tif_files = [
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000000000-0000000000.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000000000-0000004096.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000000000-0000008192.tif",

        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000004096-0000000000.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000004096-0000004096.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000004096-0000008192.tif",

        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000008192-0000000000.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000008192-0000004096.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000008192-0000008192.tif",

        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000012288-0000000000.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000012288-0000004096.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000012288-0000008192.tif",

        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000016384-0000000000.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000016384-0000004096.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000016384-0000008192.tif",

        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000020480-0000004096.tif",
        #     parent_folder / "soybeans_IL_2014_vi_stack_minimal-0000020480-0000008192.tif",
        # ]

        # # Remove accidental duplicate paths
        # tif_files = list(dict.fromkeys(tif_files))

        if not tif_files:
            raise FileNotFoundError("No TIFF files were provided or found.")

        # Check whether every listed file exists
        missing_files = [
            str(path)
            for path in tif_files
            if not path.exists()
        ]

        if missing_files:
            raise FileNotFoundError(
                "The following TIFF files do not exist:\n"
                + "\n".join(missing_files)
            )

        print(f"Number of TIFF files found: {len(tif_files)}")

        for index, path in enumerate(tif_files, start=1):
            print(f"{index:>3}. {path}")


        # ============================================================
        # INSPECT AND VALIDATE INPUT FILES
        # ============================================================

        input_information = []

        for tif_path in tif_files:
            with rasterio.open(tif_path) as src:
                input_information.append(
                    {
                        "file": tif_path.name,
                        "width": src.width,
                        "height": src.height,
                        "band_count": src.count,
                        "crs": src.crs,
                        "dtype": src.dtypes,
                        "nodata": src.nodata,
                        "resolution": src.res,
                        "bounds": src.bounds,
                        "band_descriptions": src.descriptions,
                    }
                )

        reference = input_information[0]

        print("Reference raster:")
        print(f"  File:              {reference['file']}")
        print(f"  CRS:               {reference['crs']}")
        print(f"  Number of bands:   {reference['band_count']}")
        print(f"  Data type:         {reference['dtype']}")
        print(f"  Resolution:        {reference['resolution']}")
        print(f"  NoData value:      {reference['nodata']}")
        print(f"  Band descriptions: {reference['band_descriptions']}")

        # Check whether the files have compatible structures.
        for info in input_information[1:]:
            if info["crs"] != reference["crs"]:
                raise ValueError(
                    f"CRS mismatch:\n"
                    f"  {reference['file']}: {reference['crs']}\n"
                    f"  {info['file']}: {info['crs']}\n"
                    "Reproject the rasters to the same CRS before mosaicking."
                )

            if info["band_count"] != reference["band_count"]:
                raise ValueError(
                    f"Band-count mismatch:\n"
                    f"  {reference['file']}: {reference['band_count']} bands\n"
                    f"  {info['file']}: {info['band_count']} bands"
                )

            if info["dtype"] != reference["dtype"]:
                raise ValueError(
                    f"Data-type mismatch:\n"
                    f"  {reference['file']}: {reference['dtype']}\n"
                    f"  {info['file']}: {info['dtype']}"
                )

            if not np.allclose(info["resolution"], reference["resolution"]):
                print(
                    f"Warning: {info['file']} has resolution "
                    f"{info['resolution']}, while the reference resolution is "
                    f"{reference['resolution']}."
                )

            if info["band_descriptions"] != reference["band_descriptions"]:
                print(
                    f"Warning: Band descriptions differ in {info['file']}.\n"
                    f"  Reference: {reference['band_descriptions']}\n"
                    f"  Current:   {info['band_descriptions']}"
                )

        print("\nValidation completed.")


        # ============================================================
        # MOSAIC THE TIFF FILES
        # ============================================================

        # "first" means that when rasters overlap, the first valid pixel
        # encountered in tif_files is retained.
        #
        # Other possible merge methods include:
        #   "last"  - use values from the last raster
        #   "min"   - use the minimum overlapping value
        #   "max"   - use the maximum overlapping value
        #   "sum"   - sum overlapping values
        #   "count" - count valid overlapping values

        merge_method = "first"

        # Open all input datasets.
        source_datasets = [rasterio.open(path) for path in tif_files]

        try:
            combined_array, combined_transform = merge(
                source_datasets,
                method=merge_method,
                nodata=source_datasets[0].nodata,
            )

            # Copy the first raster's profile as the basis of the output profile.
            combined_profile = source_datasets[0].profile.copy()

            # Preserve band names/descriptions from the first raster.
            band_descriptions = source_datasets[0].descriptions

            # Update dimensions and transform for the combined raster.
            combined_profile.update(
                {
                    "driver": "GTiff",
                    "height": combined_array.shape[1],
                    "width": combined_array.shape[2],
                    "count": combined_array.shape[0],
                    "transform": combined_transform,
                    "compress": "deflate",
                    "tiled": True,
                    "BIGTIFF": "IF_SAFER",
                }
            )

        finally:
            # Always close the source TIFF files.
            for dataset in source_datasets:
                dataset.close()

        print("TIFF files successfully combined in memory.")
        print(f"Combined array shape: {combined_array.shape}")
        print(
            "Array dimensions: "
            "(number of bands, raster height, raster width)"
        )


        # ============================================================
        # DISPLAY COMBINED RASTER INFORMATION
        # ============================================================

        nodata_value = combined_profile.get("nodata")

        print("COMBINED RASTER INFORMATION")
        print("=" * 65)
        print(f"Driver:             {combined_profile['driver']}")
        print(f"CRS:                {combined_profile['crs']}")
        print(f"Band count:         {combined_profile['count']}")
        print(f"Width:              {combined_profile['width']:,} pixels")
        print(f"Height:             {combined_profile['height']:,} pixels")
        print(f"Data type:          {combined_profile['dtype']}")
        print(f"NoData value:       {nodata_value}")
        print(f"Pixel resolution:   {combined_transform.a}, {abs(combined_transform.e)}")
        print(f"Affine transform:   {combined_transform}")
        print(f"Band descriptions:  {band_descriptions}")

        # Calculate spatial bounds from the combined transform.
        left = combined_transform.c
        top = combined_transform.f
        right = left + combined_profile["width"] * combined_transform.a
        bottom = top + combined_profile["height"] * combined_transform.e

        print(f"Bounds:")
        print(f"  Left:   {left}")
        print(f"  Bottom: {bottom}")
        print(f"  Right:  {right}")
        print(f"  Top:    {top}")

        print("\nBAND STATISTICS")
        print("=" * 65)

        for band_index in range(combined_array.shape[0]):
            band = combined_array[band_index]

            # Construct a valid-pixel mask.
            if nodata_value is not None:
                if np.isnan(nodata_value):
                    valid_mask = np.isfinite(band)
                else:
                    valid_mask = (
                        np.isfinite(band)
                        & (band != nodata_value)
                    )
            else:
                valid_mask = np.isfinite(band)

            valid_values = band[valid_mask]

            description = band_descriptions[band_index]
            band_name = description or f"Band {band_index + 1}"

            print(f"\nBand {band_index + 1}: {band_name}")
            print(f"  Total pixels:      {band.size:,}")
            print(f"  Valid pixels:      {valid_values.size:,}")
            print(f"  Missing pixels:    {band.size - valid_values.size:,}")

            if valid_values.size > 0:
                print(f"  Minimum:           {np.min(valid_values):.6g}")
                print(f"  Maximum:           {np.max(valid_values):.6g}")
                print(f"  Mean:              {np.mean(valid_values):.6g}")
                print(f"  Median:            {np.median(valid_values):.6g}")
                print(f"  Standard deviation:{np.std(valid_values):.6g}")
                print(f"  2nd percentile:    {np.percentile(valid_values, 2):.6g}")
                print(f"  98th percentile:   {np.percentile(valid_values, 98):.6g}")
            else:
                print("  No valid pixel values were found.")



        output_tif = Path(
            f"/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/yield_labels/multistate_pseudo/{YEAR}/soybeans_{STATE}_{YEAR}_pseudo_yield.tif"
        )

        # Create the parent directory when it does not already exist.
        output_tif.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with rasterio.open(
            output_tif,
            "w",
            **combined_profile,
        ) as dst:
            dst.write(combined_array)

            # Restore the original band descriptions.
            for band_number, description in enumerate(
                band_descriptions,
                start=1,
            ):
                if description:
                    dst.set_band_description(
                        band_number,
                        description,
                    )

        print(f"Combined TIFF saved successfully:\n{output_tif}")

Number of files found: 87
Files found:
/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/Intermediate/soybeans_MN_2023_27001_Aitkin_pseudo_yield.tif
/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/Intermediate/soybeans_MN_2023_27003_Anoka_pseudo_yield.tif
/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/Intermediate/soybeans_MN_2023_27005_Becker_pseudo_yield.tif
/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/Intermediate/soybeans_MN_2023_27007_Beltrami_pseudo_yield.tif
/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/Intermediate/soybeans_MN_2023_27009_Benton_pseudo_yield.tif
/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/pixel_distributed_yield/Intermediate/soybeans_MN_2023_27011_Big_pseudo_y

PLot:

In [ ]:
# # ============================================================
# # PLOT THE COMBINED RASTER BEFORE SAVING
# # ============================================================

# nodata_value = combined_profile.get("nodata")

# # Plot no more than six bands at once to keep the figure readable.
# number_of_bands_to_plot = min(combined_array.shape[0], 6)

# fig, axes = plt.subplots(
#     nrows=number_of_bands_to_plot,
#     ncols=1,
#     figsize=(12, 5 * number_of_bands_to_plot),
#     constrained_layout=True,
# )

# # Ensure axes is iterable when there is only one band.
# axes = np.atleast_1d(axes)

# extent = plotting_extent(
#     combined_array[0],
#     combined_transform,
# )

# for band_index, axis in enumerate(axes):
#     band = combined_array[band_index].astype("float64")

#     # Convert NoData pixels to NaN so that they do not affect
#     # statistics or visualization.
#     if nodata_value is not None:
#         if np.isnan(nodata_value):
#             band[~np.isfinite(band)] = np.nan
#         else:
#             band[band == nodata_value] = np.nan

#     band[~np.isfinite(band)] = np.nan
#     valid_values = band[np.isfinite(band)]

#     description = band_descriptions[band_index]
#     band_name = description or f"Band {band_index + 1}"

#     if valid_values.size == 0:
#         axis.text(
#             0.5,
#             0.5,
#             "No valid data",
#             ha="center",
#             va="center",
#             transform=axis.transAxes,
#         )
#         axis.set_title(band_name)
#         axis.set_axis_off()
#         continue

#     # Percentile stretch for a more informative display.
#     lower_limit, upper_limit = np.percentile(
#         valid_values,
#         [2, 98],
#     )

#     # Handle a constant-value band.
#     if lower_limit == upper_limit:
#         lower_limit = np.min(valid_values)
#         upper_limit = np.max(valid_values)

#     image = axis.imshow(
#         band,
#         extent=extent,
#         vmin=lower_limit,
#         vmax=upper_limit,
#     )

#     axis.set_title(
#         f"{band_name}\n"
#         f"Display range: {lower_limit:.4g} to {upper_limit:.4g}"
#     )
#     axis.set_xlabel("X coordinate")
#     axis.set_ylabel("Y coordinate")

#     colorbar = fig.colorbar(
#         image,
#         ax=axis,
#         shrink=0.8,
#     )
#     colorbar.set_label(band_name)

# plt.show()

# if combined_array.shape[0] > number_of_bands_to_plot:
#     print(
#         f"The raster contains {combined_array.shape[0]} bands. "
#         f"Only the first {number_of_bands_to_plot} were plotted."
#     )

In [ ]:
# ============================================================
# SAVE THE COMBINED TIFF
# ============================================================

# Change this to the desired output location.
# output_tif = Path(
#    f"/home/cholab/LabMembers/Samar/EO-based-yield-prediction/data_preparation/data/year_state_VI_summary/{YEAR}/soybeans_{STATE}_{YEAR}_vi_stack_minimal_combined.tif"
# )



## End
